# Estudo In Silico - Etapa 2: Dados Reais de Rim (Lesão vs. Regeneração)
### Pipeline de download e análise de biópsias reais de rim

Neste notebook vamos:
1. Baixar uma matriz pública real de células de rim (*scRNA-seq*).
2. Identificar os tipos celulares renais específicos (Túbulo Proximal, Alça de Henle, Podócitos).
3. Comparar a ativação de genes de estresse e reparo (*SOX9*, *HAVCR1/KIM-1*, *HNF4A*).

In [ ]:
# 1. Instalação das ferramentas científicas
!pip install --quiet scanpy anndata matplotlib seaborn pandas leidenalg igraph

In [ ]:
# 2. Carregar bibliotecas
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=110, facecolor='white', frameon=False)
print("Bibliotecas carregadas com sucesso!")

### 3. Download do Dataset Real de Rim
Utilizamos um conjunto de dados real de células renais curado e padronizado em formato `.h5ad` (AnnData).

In [ ]:
# Download direto em alta velocidade do Atlas de Rim Humano Real (CZ CELLxGENE / Tabula Sapiens)
import os
import scanpy as sc

arquivo_destino = "kidney_real_data.h5ad"
url_rim = "https://datasets.cellxgene.cziscience.com/65ca6e36-73b0-4c88-b0f3-7b23b48844ad.h5ad"

if not os.path.exists(arquivo_destino):
    print("Baixando Atlas Real de Células Renais (400 MB na velocidade do Google Colab)...")
    !wget -q --show-progress -O kidney_real_data.h5ad "$url_rim"

print("Carregando a matriz de células renais...")
adata = sc.read_h5ad(arquivo_destino)

print(f"Total de células renais: {adata.n_obs}")
print(f"Total de genes analisados: {adata.n_vars}")

### 4. Visualizar o Mapa de Células Renais

In [ ]:
# Se o UMAP já estiver calculado no dataset, plotamos diretamente; senão calculamos:
if 'X_umap' not in adata.obsm:
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20)
    sc.tl.leiden(adata, resolution=0.5, flavor='igraph', n_iterations=2)
    sc.tl.umap(adata)

# Escolhe a coluna de anotação celular disponível
coluna_grupo = 'cell_type' if 'cell_type' in adata.obs else ('louvain' if 'louvain' in adata.obs else 'leiden')
sc.pl.umap(adata, color=[coluna_grupo], title="Atlas de Células Renais (Lesão & Regeneração)")

### 5. Análise dos Interruptores Genéticos de Regeneração e Fibrose
Vamos mapear a atividade dos genes que determinam se o rim se regenera ou cicatriza com fibrose.

In [ ]:
# Lista de genes marcadores do estudo de regeneração renal
# Vamos checar quais desses genes existem na matriz:
genes_alvo = ["SOX9", "PAX2", "WNT4", "HAVCR1", "HNF4A", "ACTA2", "TGFB1"]
genes_presentes = [g for g in genes_alvo if g in adata.var_names]

print(f"Genes do painel de regeneração encontrados nesta amostra: {genes_presentes}")

if genes_presentes:
    # Gera gráfico de intensidade da expressão (amarelo = hiperativo, roxo = desligado)
    sc.pl.umap(adata, color=genes_presentes, color_map='magma', ncols=3)
else:
    # Caso os nomes de genes usem outro formato, listamos os 10 genes mais expressos
    sc.pl.highest_expr_genes(adata, n_top=15)